# PRC1.3: Transfer Learning via Fine-tuning for Image Classification (TASK 3)

**Goals:**
This notebook is part of the first programming assignment (PRC1) for the course "Deep Learning for Visual Signal Processing I." In this notebook, you are required to undertake a series of tasks listed in the "2.TASKS" section.

**Learning Objectives:**
* Understand the concept of transfer learning: Learn how pre-trained deep learning models can be adapted to new classification tasks with minimal training effort.
* Implement model fine-tuning techniques: Modify and train a pre-trained model by freezing and unfreezing layers, replacing classifier heads, and optimizing hyperparameters.
* Evaluate and compare model performance: Analyze the impact of different fine-tuning strategies on model accuracy and efficiency, and explore best practices for transfer learning in deep learning applications.


**Expected Outcomes:**
* Notebooks: Generate separate notebooks for each experiment conducted during this task.
* Report: Submit a concise report (no more than two pages) that adheres to the specified course format, summarizing your findings and analyses.

**Estimated Completion Time:** The tasks are designed to be completed within an estimated timeframe of 2-3 hours when using GPU acceleration.

---

Author1: Sabbatini, Andrea (andrea.sabbatini@estudiante.uam.es)

Author2: Hamdy, Adham (adham.hamdy@estudiante.uam.es)

Author3: Ciurescu, Irina Alexandra (irinaa.ciurescu@estudiante.uam.es)

---
###### Course: Deep Learning for Visual Signal Processing I
###### Master in [Artificial Intelligence for Image Processing and Computer Vision (IPCVai)](https://ipcv.eu/)
######  [Escuela Politécnica Superior](https://www.uam.es/EPS/Home.htm), [Universidad Autónoma de Madrid](https://www.uam.es/)


# 1. Codebase

We use the provided codebase in the notebooks to develop this assignment. It contains essential scripts to access the dataset and establish the training partitions required for the tasks.

### 1-1.Install the required libraries

Check versions of installed software for this tutorial

* Python 3.10 or above
* Pytorch 2.5.1+cu121 or above
* Torchvision 0.20.1+cu121 or above

In [ ]:
import torch
import numpy as np

# Reproducibility (note: full determinism on GPU may require extra flags)
torch.manual_seed(0)
np.random.seed(0)

!python --version

import torch
print(torch.__version__)

import torchvision
print(torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Python 3.12.12
2.9.0+cu126
0.24.0+cu126
Using device: cuda


### 1-2.Utility Functions (do not modify)

This cell installs and imports a set of shared utility functions used throughout the practical assignments.  
These utilities provide common functionality for:

- Model performance evaluation (overall accuracy, per-class accuracy, reports)
- Dataset inspection and class distribution analysis
- Dataset filtering and class selection
- Visualization of class imbalance

The package is maintained in a centralized GitHub repository to ensure consistency across all notebooks.  
Please **do not edit this cell**, as later sections of the tutorial rely on these functions.

In [ ]:
!pip install -q git+https://github.com/jcsma/dlvsp-utils.git

from dlvsp_utils.metrics import (
    calculate_accuracy,
    compute_accuracy_stats,
    compute_accuracy_per_class,
    print_accuracy_report,
)

from dlvsp_utils.data import (
    count_samples_dataset,
    select_classes_dataset,
    inspect_dataset_classes,
)
from dlvsp_utils.viz import (
    plot_class_distribution
)

print("✅ DLVSP utils imported: performance analysis and dataset handling ready!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ DLVSP utils imported: performance analysis and dataset handling ready!


# 2. Loading the dataset for the Classification Task
In this section we load the CIFAR-10 dataset with some basic preprocessing and chose a subset of class.

### 2-1.Basic Preprocessing
We define a basic preprocessing obtained by the tutorial.

In [ ]:
from torchvision import datasets, transforms

# Define a basic transformation when loading the dataset
transform = transforms.Compose([
    #transforms.Resize(224), #for selective fine-tuning, we will get better performance if we match the training image size of popular backbones for ImageNet (224x224)
    transforms.ToTensor(),  # Convert images to tensor format
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]#we get better performance if we match the training image normalization of popular backbones for ImageNet
    )
])

### 2-2.Load CIFAR-10
In this cell, we load the CIFAR-10 dataset (training and test splits) and apply the preprocessing.

In [ ]:
# Load CIFAR-10 data
train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
class_names = train_full.classes
print(class_names)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### 2-3.Subset of classes
We choose a subset of 4 classes and load the new datasets.

In [ ]:
# Define the classes to be used (e.g., 4 classes from CIFAR-10)
selected_class_names = ['dog', 'cat', 'airplane', 'bird']   # e.g., ['cat','dog'] or 10 classes

# Select subset of classes from CIFAR10
train_dataset_cifar, class_names = select_classes_dataset(train_full, selected_class_names)
test_dataset_cifar, _ = select_classes_dataset(test_full, selected_class_names)

num_classes = len(class_names)
print("Selected classes:", class_names)
print("num_classes:", num_classes)
inspect_dataset_classes(train_dataset_cifar, class_names=class_names, header="\nTRAIN:")
inspect_dataset_classes(test_dataset_cifar, class_names=class_names, header="\nTEST:")
print("✅ Datasets are ready!")

Selected classes: ['dog', 'cat', 'airplane', 'bird']
num_classes: 4

TRAIN:
Classes present:
  Class 0 (dog): 5000 samples
  Class 1 (cat): 5000 samples
  Class 2 (airplane): 5000 samples
  Class 3 (bird): 5000 samples

TEST:
Classes present:
  Class 0 (dog): 1000 samples
  Class 1 (cat): 1000 samples
  Class 2 (airplane): 1000 samples
  Class 3 (bird): 1000 samples
✅ Datasets are ready!


# 3. Linear head
In this section we train a `ResNet18` initialized with ImageNet weights (full fine-tuning), adding a linear final classification head.

### 3-1.Define the ResNet Architecture
We define a `ResNet-18` model using the standard architecture and setting the parameters for full fine-tuning.
The final classification layer is replaced with a linear head.

In [ ]:
import torch.nn as nn
from torchvision import models

# ResNet18 architecture
model_light = models.resnet18(weights='IMAGENET1K_V1')

# Create a new classification head for CIFAR-10
num_ftrs = model_light.fc.in_features

# Create a new classification head
# Here the size of each output sample is set to the number of selected classes
model_light.fc = nn.Linear(num_ftrs, num_classes)
for param in model_light.parameters():
    param.requires_grad = True

print("✅ResNet18 architecture with linear head defined!")

num_params = sum(p.numel() for p in model_light.parameters())
print("Number of parameters: "+str(num_params))

✅ResNet18 architecture with linear head defined!
Number of parameters: 11178564


### 3-2.Training setup
In this cell, we define the main training components: hyperparameters, data loaders, the model, the loss function, and the optimizer.

In [ ]:
# Training setup for ResNet18 architecture with linear head
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_workers=4
num_epochs = 10
batch_size=64
lr=0.001
weight_decay=1e-5

# Setup dataloaders
train_loader = DataLoader(train_dataset_cifar, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset_cifar, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Setup model, loss, and optimizer
model_light = model_light.to(device)  # instantiate custom CNN
criterion_light = nn.CrossEntropyLoss()
optimizer_light = optim.Adam(model_light.parameters(), lr=lr, weight_decay=weight_decay)

print("✅ Training setup ready for ResNet18 with linear head!")

✅ Training setup ready for ResNet18 with linear head!


In [ ]:
import time

# Training loop for the ResNet18
print(f"Running training for ResNet18 network (pretrained, full fine-tuning, linear head) in {device} mode for {num_epochs} epochs")

# Start the timer
start_time = time.time()

for epoch in range(num_epochs):
    model_light.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_light.zero_grad()
        outputs = model_light(images)
        loss = criterion_light(outputs, labels)
        loss.backward()
        optimizer_light.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_light.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_light)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_light)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# Stop the timer and print training time
end_time = time.time()
duration = end_time - start_time
print("\nTotal training time: "+str(duration)+" seconds")

# Per-class accuracy
print_accuracy_report(train_labels, train_preds,
                     class_names=class_names,
                     header="TRAIN - Accuracy per class (#train samples /total):",
                     samples_per_class=count_samples_dataset(train_dataset_cifar, num_classes=num_classes))

print_accuracy_report(test_labels, test_preds,
                     class_names=class_names,
                     header="\nTEST - Accuracy per class (#test samples /total):",
                     samples_per_class=count_samples_dataset(test_dataset_cifar, num_classes=num_classes))
print("✅ Training completed!")

Running training for ResNet18 network (pretrained, full fine-tuning, linear head) in cuda mode for 10 epochs
Epoch [1/10], Loss: 0.7140, Train Accuracy: 82.02%, Test Accuracy: 77.83%
Epoch [2/10], Loss: 0.4955, Train Accuracy: 83.88%, Test Accuracy: 77.08%
Epoch [3/10], Loss: 0.4011, Train Accuracy: 87.47%, Test Accuracy: 79.60%
Epoch [4/10], Loss: 0.3382, Train Accuracy: 91.78%, Test Accuracy: 80.72%
Epoch [5/10], Loss: 0.2766, Train Accuracy: 94.90%, Test Accuracy: 82.90%
Epoch [6/10], Loss: 0.2158, Train Accuracy: 93.97%, Test Accuracy: 80.42%
Epoch [7/10], Loss: 0.1796, Train Accuracy: 96.23%, Test Accuracy: 80.03%
Epoch [8/10], Loss: 0.1419, Train Accuracy: 96.16%, Test Accuracy: 80.85%
Epoch [9/10], Loss: 0.1195, Train Accuracy: 97.75%, Test Accuracy: 80.75%
Epoch [10/10], Loss: 0.1058, Train Accuracy: 98.18%, Test Accuracy: 81.47%

Total training time: 99.09657144546509 seconds
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 98.18%
Mean

### 3-3.Train the model
Run this cell to train **ResNet-18** (with `weights='IMAGENET1K_V1'`, i.e., **ImageNet pretraining** and linear head) for num_epochs epochs using the training loader. After each epoch, we switch to evaluation mode to report train and test accuracy, which helps monitor learning progress and generalization.

At the end, we print an **accuracy report per class** (train and test). This is especially useful to spot classes that the model struggles with (often visible as lower per-class accuracy, even when overall accuracy looks decent).

Moreover, we print the total training time.

# 4. Deeper head
In this section we train a `ResNet18` initialized with ImageNet weights (full fine-tuning), adding a deeper final head.

### 4-1.Define the ResNet18 Architecture
We define a `ResNet-18` model using the standard architecture and setting the parameters for full fine-tuning.
The final classification layer is replaced with a Linear → ReLU → Dropout → Linear head.

In [ ]:
import torch.nn as nn
from torchvision import models

# ResNet architecture
model_heavy = models.resnet18(weights='IMAGENET1K_V1')

# Create a new classification head for CIFAR-10
num_ftrs = model_heavy.fc.in_features

# Create a new classification head
# Here the size of each output sample is set to the number of selected classes
model_heavy.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512, num_classes)
)
for param in model_heavy.parameters():
    param.requires_grad = True

print("✅ResNet18 architecture with deeper head defined!")

num_params = sum(p.numel() for p in model_heavy.parameters())
print("Number of parameters: "+str(num_params))

✅ResNet18 architecture with deeper head defined!
Number of parameters: 11441220


### 4-2.Training setup
In this cell, we define the main training components: hyperparameters (the same as before), data loaders, the model, the loss function, and the optimizer.

In [ ]:
# Training setup for ResNet18 architecture with deeper head
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
num_workers=4
num_epochs = 10
batch_size=64
lr=0.001
weight_decay=1e-5

# Setup dataloaders
train_loader = DataLoader(train_dataset_cifar, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset_cifar, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Setup model, loss, and optimizer
model_heavy = model_heavy.to(device)  # instantiate custom CNN
criterion_heavy = nn.CrossEntropyLoss()
optimizer_heavy = optim.Adam(model_heavy.parameters(), lr=lr, weight_decay=weight_decay)

print("✅ Training setup ready for ResNet18 with deeper head!")

✅ Training setup ready for ResNet18 with deeper head!


### 4-3.Train the model
Run this cell to train **ResNet-18** (with `weights='IMAGENET1K_V1'`, i.e., **ImageNet pretraining** and deeper head) for num_epochs epochs using the training loader. After each epoch, we switch to evaluation mode to report train and test accuracy, which helps monitor learning progress and generalization.

At the end, we print an **accuracy report per class** (train and test). This is especially useful to spot classes that the model struggles with (often visible as lower per-class accuracy, even when overall accuracy looks decent).

Moreover, we print the total training time.

In [ ]:
import time

# Training loop for the ResNet18
print(f"Running training for ResNet18 network (pretrained, full fine-tuning, deeper head) in {device} mode for {num_epochs} epochs")

# Start the timer
start_time = time.time()

for epoch in range(num_epochs):
    model_heavy.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_heavy.zero_grad()
        outputs = model_heavy(images)
        loss = criterion_heavy(outputs, labels)
        loss.backward()
        optimizer_heavy.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # evaluate epoch results
    model_heavy.eval()
    train_accuracy, train_labels, train_preds = calculate_accuracy(train_loader, model_heavy)
    test_accuracy, test_labels, test_preds = calculate_accuracy(test_loader, model_heavy)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')

# Stop the timer and print training time
end_time = time.time()
duration = end_time - start_time
print("\nTotal training time: "+str(duration)+" seconds")

# Per-class accuracy
print_accuracy_report(train_labels, train_preds,
                     class_names=class_names,
                     header="TRAIN - Accuracy per class (#train samples /total):",
                     samples_per_class=count_samples_dataset(train_dataset_cifar, num_classes=num_classes))

print_accuracy_report(test_labels, test_preds,
                     class_names=class_names,
                     header="\nTEST - Accuracy per class (#test samples /total):",
                     samples_per_class=count_samples_dataset(test_dataset_cifar, num_classes=num_classes))
print("✅ Training completed!")

Running training for ResNet18 network (pretrained, full fine-tuning, deeper head) in cuda mode for 10 epochs
Epoch [1/10], Loss: 0.7289, Train Accuracy: 78.03%, Test Accuracy: 74.05%
Epoch [2/10], Loss: 0.5547, Train Accuracy: 84.42%, Test Accuracy: 77.92%
Epoch [3/10], Loss: 0.4523, Train Accuracy: 88.11%, Test Accuracy: 80.45%
Epoch [4/10], Loss: 0.3758, Train Accuracy: 91.62%, Test Accuracy: 81.25%
Epoch [5/10], Loss: 0.3069, Train Accuracy: 91.86%, Test Accuracy: 80.10%
Epoch [6/10], Loss: 0.2388, Train Accuracy: 94.70%, Test Accuracy: 80.65%
Epoch [7/10], Loss: 0.2657, Train Accuracy: 94.75%, Test Accuracy: 80.70%
Epoch [8/10], Loss: 0.1668, Train Accuracy: 96.90%, Test Accuracy: 81.03%
Epoch [9/10], Loss: 0.1242, Train Accuracy: 96.16%, Test Accuracy: 79.50%
Epoch [10/10], Loss: 0.1047, Train Accuracy: 96.67%, Test Accuracy: 80.67%

Total training time: 99.32901263237 seconds
TRAIN - Accuracy per class (#train samples /total):
Overall accuracy (micro, all samples): 96.67%
Mean pe